In [ ]:
import os
import re
from dotenv import load_dotenv
from langchain.chat_models import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain.prompts import PromptTemplate
from datasets import load_dataset
from langchain.llms.base import BaseLLM
from langchain.chat_models.base import BaseChatModel
from langchain.schema import HumanMessage
import multiprocessing
from functools import partial
import time

load_dotenv()

In [ ]:
dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

def init_llms():
    return {
        # "chatgpt": ChatOpenAI(
        #     model_name="gpt-4o",
        #     openai_api_key=os.getenv("OPENAI_API_KEY")
        # ),
        # "claude": ChatAnthropic(
        #     model_name="claude-3-5-sonnet-20240620",
        #     anthropic_api_key=os.getenv("ANTHROPIC_API_KEY")
        # ),
        "deepseek": ChatDeepSeek(
            model="deepseek-reasoner",
            api_key=os.getenv("DEEPSEEK_API_KEY")
        )
    }

In [ ]:
# # Prompt Template
# prompt_template = PromptTemplate(
#     input_variables=["problem_statement", "file_content", "file_path"],
#     template="""
# We are solving the following issue:
# --- BEGIN ISSUE ---
# {problem_statement}
# --- END ISSUE ---

# Below is the relevant file:
# --- BEGIN FILE ---
# ```
# {file_content}
# ```
# --- END FILE ---

# Please fix the issues in the code. Instead of providing the complete fixed code, respond ONLY with a Git diff that shows your changes.

# You are a professional Software Engineer tasked with fixing a bug in the repository. Above is the user's description of the bug, which should include details such as steps to reproduce, expected behavior, and actual behavior. If the description is unclear or lacks sufficient detail, please ask the user for clarification.

# Your goal is to identify and correct the bug while adhering to the following guidelines:

# 1. Understand the Bug:
#    - Carefully read the bug description provided by the user.
#    - If available, review any error messages, stack traces, or logs that might help pinpoint the issue.
#    - Consider looking at related issues or pull requests in the repository for additional context or potential solutions.

# 2. Locate the Bug:
#    - Based on the bug description and any additional information, identify the most likely functions or sections of code that are causing the issue.
#    - Use the context of the bug to narrow down the possible locations.

# 3. Fix the Bug:
#    - Implement a fix that directly addresses the root cause of the bug.
#    - Ensure that your fix is minimal and does not introduce new bugs or regressions.
#    - Do not alter the overall architecture of the repository.
#    - Maintain the existing input types, output types, and the number of parameters for all functions.
#    - Ensure that your code changes are consistent with the coding style and conventions used in the repository.

# 4. Verify the Fix:
#    - Test your changes to confirm that the bug is resolved.
#    - Check that your fix does not cause any new issues or break existing functionality.

# Output format:
# 1. Use the Git diff format to show the changes.
# 2. Start with "--- a/{file_path}" for the original code.
# 3. Start with "+++ b/{file_path}" for the modified code.
# 4. Use @@ to indicate the line numbers and context.
# 5. Use - to show removed lines and + to show added lines.
# 6. Ensure the output is a valid Git diff format so I can easily extract it.

# Your response should ONLY contain the Git diff and nothing else.
# """
# )

In [ ]:
# Prompt Template
prompt_template = PromptTemplate(
    input_variables=["problem_statement", "file_content", "file_path"],
    template="""
We are solving the following issue:
--- BEGIN ISSUE ---
{problem_statement}
--- END ISSUE ---

Below is the relevant file:
--- BEGIN FILE ---
```
{file_content}
```
--- END FILE ---

Please fix the issues in the code. Instead of providing the complete fixed code, respond ONLY with a Git diff that shows your changes.

Output format:
1. Use the Git diff format to show the changes.
2. Start with "--- a/{file_path}" for the original code.
3. Start with "+++ b/{file_path}" for the modified code.
4. Use @@ to indicate the line numbers and context.
5. Use - to show removed lines and + to show added lines.
6. Ensure the output is a valid Git diff format so I can easily extract it.

Your response should ONLY contain the Git diff and nothing else.
"""
)

In [ ]:
def extract_modified_file_path(patch):
    """Extracts the modified file path from the first line of a Git diff."""
    match = re.search(r'diff --git a/(.*?) b/', patch)
    return match.group(1) if match else None

def prepare_task(task):
    """Prepares a task by extracting necessary information and file content."""
    instance_id = task["instance_id"]
    problem_statement = task["problem_statement"]
    patch = task["patch"]
    
    # Extract file path from patch
    file_path = extract_modified_file_path(patch)
    if not file_path:
        print(f"[Warning] No file path found for {instance_id}")
        return None
    
    # Read file content
    file_full_path = f"./codebases/{instance_id}/{file_path}"
    if not os.path.exists(file_full_path):
        print(f"[Error] File not found: {file_full_path}")
        return None
    
    with open(file_full_path, "r", encoding="utf-8") as f:
        file_content = f.read()
    
    return {
        "instance_id": instance_id,
        "problem_statement": problem_statement,
        "file_content": file_content,
        "file_path": file_path
    }

In [ ]:
def process_task(prepared_task, llm_name, llm_dict):
    """Processes a single prepared task using the specified LLM."""
    if not prepared_task:
        return
    
    try:
        instance_id = prepared_task["instance_id"]
        
        # Format prompt
        prompt = prompt_template.format(
            problem_statement=prepared_task["problem_statement"], 
            file_content=prepared_task["file_content"],
            file_path=prepared_task["file_path"]
        )
        
        # Get response from LLM
        llm = llm_dict[llm_name]
        if isinstance(llm, BaseChatModel):
            response = llm.invoke([HumanMessage(content=prompt)]).content
        elif isinstance(llm, BaseLLM):
            response = llm.predict(prompt)
        else:
            raise ValueError(f"Unknown LLM type for {llm_name}")
        
        # The LLM should directly return a diff, so we store it as is
        # Strip out any potential code block markers
        diff_output = response.strip()
        if diff_output.startswith("```") and diff_output.endswith("```"):
            diff_output = diff_output[3:-3].strip()
        
        # Save the diff
        output_path = f"./test_outputs/{llm_name}/direct_format_detailed/{instance_id}.diff"
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(diff_output)
        
        print(f"[Success] Diff saved: {output_path}")
        return instance_id, True
    except Exception as e:
        print(f"[Error] Failed to process {instance_id} with {llm_name}: {e}")
        return instance_id, False

In [ ]:
def worker_init(llm_name):
    """Initialize worker process with its own LLM instance."""
    global worker_llm_dict
    worker_llm_dict = init_llms()
    print(f"Worker initialized with {llm_name}")

def worker_process(prepared_task, llm_name):
    """Worker function that processes a task with the global LLM dict."""
    return process_task(prepared_task, llm_name, worker_llm_dict)

In [ ]:
llm_name = 'deepseek'

start_time = time.time()

# Prepare all tasks first (I/O operations)
prepared_tasks = []
for task in dataset:
    prepared_task = prepare_task(task)
    if prepared_task:
        prepared_tasks.append(prepared_task)

print(f"Prepared {len(prepared_tasks)} tasks in {time.time() - start_time:.2f} seconds")

# Process tasks in parallel
num_processes = min(multiprocessing.cpu_count(), 8)  # Limiting to prevent API throttling
# num_processes = 10

print(f"Processing with {num_processes} processes")

# Use a process pool with initialization
with multiprocessing.Pool(
    processes=num_processes,
    initializer=worker_init,
    initargs=(llm_name,)
) as pool:
    # Create a partial function with the llm_name already set
    process_func = partial(worker_process, llm_name=llm_name)
    
    # Map the function to all tasks
    results = pool.map(process_func, prepared_tasks)

# Count successes
success_count = sum(1 for result in results if result and result[1])

print(f"Completed processing {len(prepared_tasks)} tasks")
print(f"Successful: {success_count}")
print(f"Failed: {len(prepared_tasks) - success_count}")
print(f"Total time: {time.time() - start_time:.2f} seconds")